# AIkenGPT JCommonsenseQA 921問 生成評価

`boxed_answer_only_model_epoch_1_lr_1e-04_easy_jmmlu_text.safetensors` で921問すべてを生成し、`\boxed{...}` 内の答えを正解文言と照合します。出力CSVには問題・正解・生出力・抽出回答・完全一致フラグを残すので、誤りや形式崩れを手動確認できます。

ColabのGPUランタイムで上から実行してください。

In [1]:
%pip install -q numpy pandas tiktoken safetensors huggingface_hub tqdm

from pathlib import Path
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = Path("/content/AIkenSGTv1_New")
DATA_REPO_DIR = Path("/content/AIkenSGTv1_easy_jmmlu")
if not (PROJECT_DIR / "mmlu_eval").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "tayama",
                    "https://github.com/HayatoHongo/AIkenSGTv1.git", str(PROJECT_DIR)], check=True)
if not DATA_REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "easy-jmmlu",
                    "https://github.com/HayatoHongo/AIkenSGTv1.git", str(DATA_REPO_DIR)], check=True)
sys.path.insert(0, str(PROJECT_DIR))


Mounted at /content/drive


In [ ]:
import csv
import re
import torch
import pandas as pd
import tiktoken
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from mmlu_eval.backends.aikengpt_backend import load_custom_checkpoint

assert torch.cuda.is_available(), "Select a Colab GPU runtime"
MODEL_REPO = "HayatoHongo/AIkenSGTv1"
MODEL_PATH = hf_hub_download(repo_id=MODEL_REPO, filename="boxed_answer_only_model_epoch_1_lr_1e-04_easy_jmmlu_text.safetensors")
model = load_custom_checkpoint(MODEL_PATH, device="cuda", dtype="float32", max_context_length=2048)
model.eval()
tokenizer = tiktoken.get_encoding("gpt2")

DATA_CSV = DATA_REPO_DIR / "easy-jmmlu/eval_data/test/jcommonsenseqa_test.csv"
with DATA_CSV.open(encoding="utf-8", newline="") as f:
    test_rows = list(csv.reader(f))
if len(test_rows) != 921 or any(len(row) != 6 for row in test_rows):
    raise ValueError(f"Expected 921 six-column test rows; got {len(test_rows)}")

instruction = r"次の問題に答えてください。選択肢の記号ではなく、正解の文言だけを \boxed{...} 形式で答えてください。"
rows = []
for i, row in enumerate(tqdm(test_rows, desc="Generate"), start=1):
    question, *tail = row
    options, label = tail[:4], tail[4].strip()
    option_map = dict(zip("ABCD", options))
    if label not in option_map:
        raise ValueError(f"Invalid answer label at row {i}: {label!r}")
    expected = option_map[label]
    prompt = (f"<USER>{instruction}\n\n問題: {question}\n"
              f"A. {options[0]}\nB. {options[1]}\nC. {options[2]}\nD. {options[3]}\n\n答え:<ASSISTANT>")
    input_ids = torch.tensor([tokenizer.encode(prompt, allowed_special="all")],
                             dtype=torch.long, device="cuda")
    generated = []
    with torch.no_grad():
        for token_id in model.generate(input_ids, max_new_tokens=32, temperature=1.0,
                                       top_k=1, reset_cache=True):
            if token_id == tokenizer.eot_token:
                break
            generated.append(token_id)
    raw = tokenizer.decode(generated).strip()
    match = re.search(r"\\boxed\{([^{}]*)\}", raw)
    answer = match.group(1).strip() if match else ""
    rows.append({
        "question": question, "A": options[0], "B": options[1], "C": options[2], "D": options[3],
        "正解ラベル": label, "正解文言": expected, "モデル生出力": raw,
        "boxed回答": answer, "完全一致": answer == expected,
    })
results = pd.DataFrame(rows)
print(f"完全一致: {results['完全一致'].sum()} / {len(results)} ({results['完全一致'].mean():.1%})")
display(results.head(20))
display(results.loc[~results["完全一致"]].head(30))


: 

## Google Driveへ保存

全921問の生成が完了した後、このセルを実行してCSVを保存します。

In [ ]:
from pathlib import Path

OUTPUT_PATH = Path("/content/drive/MyDrive/aikengpt_results/aikengpt_model_v2_jcommonsenseqa_generation_921.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"保存しました: {OUTPUT_PATH} ({len(results)}問)")